# 05 — Normalisation et canonicalisation des entreprises

Cette étape lit la sortie de `04_programme_enrichment.ipynb`, normalise syntaxiquement les noms
d'entreprise, détecte des doublons probables par similarité TF-IDF (sans jamais fusionner
automatiquement), applique uniquement les fusions explicitement validées dans
`data/reference/company_merge_decisions.csv`, puis regroupe les entreprises canoniques en familles
analytiques.

Toute la logique métier (normalisation, détection de candidats, canonicalisation Union-Find,
règles de familles) vit dans `src/stages_cleaning/company.py` ; ce notebook se limite au
chargement, à l'appel des fonctions métier, à l'audit lisible et à l'export.

**Ce notebook est l'unique producteur du dataset final** : `data/processed/stages_efrei_final.csv`.

## 1. Imports et chemins

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from stages_cleaning.company import (
    apply_company_canonicalization, assign_company_family, build_company_candidates,
    enrich_company_normalization, reconcile_merge_decisions, validate_decisions,
)
from stages_cleaning.config import load_csv_config

INPUT_PATH = ROOT / "data" / "interim" / "stages_efrei_with_programme.csv"
OUTPUT_PATH = ROOT / "data" / "processed" / "stages_efrei_final.csv"
TRACE_PATH = ROOT / "data" / "interim" / "company_normalization_trace.csv"
CANDIDATES_PATH = ROOT / "data" / "interim" / "company_merge_candidates.csv"
DECISIONS_PATH = ROOT / "data" / "reference" / "company_merge_decisions.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
DECISIONS_PATH.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_PATH, dtype="string", encoding="utf-8-sig", keep_default_na=False).replace("", pd.NA)
n_before_rows = len(df)
print("Shape entrée :", df.shape)

## 2. Normalisation syntaxique et contrôle qualité

In [ ]:
df = enrich_company_normalization(df)
print(df["entreprise_qualite"].value_counts(dropna=False))
print("Valeurs distinctes brutes :", df["entreprise"].nunique(dropna=True))
print("Valeurs distinctes normalisées :", df["entreprise_normalisee"].nunique(dropna=True))
display(df.loc[df["entreprise_qualite"].ne("ok"), ["entreprise", "entreprise_normalisee", "entreprise_qualite"]].drop_duplicates().sort_values("entreprise_qualite"))

## 3. Détection de candidats (TF-IDF + plus proches voisins)

In [ ]:
candidates = build_company_candidates(df)
print("Paires candidates :", len(candidates))
candidates_review = candidates.loc[candidates["suggestion"].isin(["merge_probable", "review"])].copy()
candidates_review["decision"] = ""
candidates_review["entreprise_canonique"] = ""
candidates_review["commentaire"] = ""
print("Paires à examiner :", len(candidates_review))
display(candidates_review.head(30))

## 4. Export des candidats et référentiel de décisions

In [ ]:
candidates_review.to_csv(CANDIDATES_PATH, index=False, encoding="utf-8-sig")
print("Candidats exportés :", CANDIDATES_PATH)

if not DECISIONS_PATH.exists():
    candidates_review.to_csv(DECISIONS_PATH, index=False, encoding="utf-8-sig")
    print("Fichier de décisions créé :", DECISIONS_PATH)
else:
    # AUDIT_FIXES.md D-4 : les nouvelles paires candidates sont ajoutées (append-only) au
    # référentiel existant ; les décisions déjà renseignées ne sont jamais touchées.
    existing_decisions = load_csv_config(DECISIONS_PATH, {"entreprise_1", "entreprise_2", "decision", "entreprise_canonique"})
    updated_decisions = reconcile_merge_decisions(candidates_review, existing_decisions)
    n_added = len(updated_decisions) - len(existing_decisions)
    if n_added:
        updated_decisions.to_csv(DECISIONS_PATH, index=False, encoding="utf-8-sig")
        print(f"{n_added} nouvelle(s) paire(s) candidate(s) ajoutée(s) au fichier de décisions (decision=\'\').")
    else:
        print("Le fichier de décisions existe déjà, aucune nouvelle paire à ajouter :", DECISIONS_PATH)

### Validation manuelle requise

Après la première exécution :

1. ouvrir `data/reference/company_merge_decisions.csv` ;
2. renseigner `decision` avec `merge`, `distinct` ou `review` ;
3. pour les lignes `merge`, renseigner `entreprise_canonique` lorsque nécessaire ;
4. enregistrer le fichier ;
5. reprendre l'exécution du notebook à partir de la cellule suivante.

Le fichier de décisions n'est pas écrasé lors des exécutions suivantes. **Tant que `decision` reste
vide, aucune fusion n'est appliquée** : `entreprise_canonique` restera égale à `entreprise_normalisee`.

## 5. Canonicalisation à partir des décisions validées

In [ ]:
decisions = load_csv_config(DECISIONS_PATH, {"entreprise_1", "entreprise_2", "decision", "entreprise_canonique"})
decisions = validate_decisions(decisions)
print(decisions["decision"].value_counts(dropna=False))
n_merge_approved = int(decisions["decision"].eq("merge").sum())
if n_merge_approved == 0:
    print("Aucune fusion validée : entreprise_canonique == entreprise_normalisee pour toutes les lignes.")

df = apply_company_canonicalization(df, decisions)
print("Entreprises normalisées :", df["entreprise_normalisee"].nunique(dropna=True))
print("Entreprises canoniques  :", df["entreprise_canonique"].nunique(dropna=True))
rows_changed = (df["entreprise_normalisee"] != df["entreprise_canonique"]).fillna(False)
print("Lignes modifiées par la fusion :", int(rows_changed.sum()))
display(df.loc[rows_changed, ["entreprise", "entreprise_normalisee", "entreprise_canonique"]].drop_duplicates().sort_values("entreprise_canonique"))

## 6. Familles analytiques

In [ ]:
df = assign_company_family(df)
print("Familles analytiques :", df["entreprise_famille"].nunique(dropna=True))
display(df["entreprise_famille"].value_counts(dropna=False).head(30))

## 7. Audit final

In [ ]:
audit = pd.DataFrame({
    "niveau": ["brut", "normalise", "canonique", "famille"],
    "valeurs_distinctes": [
        df["entreprise"].nunique(dropna=True),
        df["entreprise_normalisee"].nunique(dropna=True),
        df["entreprise_canonique"].nunique(dropna=True),
        df["entreprise_famille"].nunique(dropna=True),
    ],
})
display(audit)

## 8. Traçabilité et export final

In [ ]:
company_trace = (
    df[["entreprise", "entreprise_normalisee", "entreprise_canonique", "entreprise_famille", "entreprise_qualite"]]
    .drop_duplicates()
    .sort_values(["entreprise_famille", "entreprise_canonique", "entreprise_normalisee"])
    .reset_index(drop=True)
)
company_trace.to_csv(TRACE_PATH, index=False, encoding="utf-8-sig")
print("Table de traçabilité exportée :", TRACE_PATH)

In [ ]:
assert len(df) == n_before_rows, "Le nombre de lignes ne doit pas changer pendant la normalisation entreprise"
assert df["stage_id"].is_unique, "stage_id doit rester unique dans le dataset final"
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print("Export final unique :", OUTPUT_PATH)
print("Shape :", df.shape)